# CNN Training Notebook

### 0. Import Library & Dataset

In [ ]:
# Standard libraries
import os
import sys
import itertools
import time
import json

from pathlib import Path

# Data Processing
import numpy as np
import pandas as pd

# Visualisasi
import matplotlib.pyplot as plt

# Evaluasi Model
from sklearn.metrics import f1_score, classification_report

# Deep Learning
import tensorflow as tf
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

# Keras Layers (Init-Inject)
from tensorflow.keras.layers import (Input, Embedding, Dense,
                                     RepeatVector, Add, LSTM, SimpleRNN)
from tensorflow.keras.models import Model

print('TF: ', tf.__version__)
print('GPU: ', tf.config.list_physical_devices('GPU'))

TF:  2.10.0
GPU:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
src = Path(os.getcwd()) # Path on this file
while not (src / 'src').exists() and src != src.parent:
    src = src.parent

sys.path.insert(0, str(src))
os.chdir(src)

In [ ]:
DATA_DIR = Path('data/flickr8k')
IMAGES_DIR = DATA_DIR / 'Images'
CAPS_CSV = DATA_DIR / 'captions.txt'
MODEL_DIR = Path('models/rnn_lstm')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEAT_CACHE  = MODEL_DIR / 'inception_features.npy'
VOCAB_PATH  = MODEL_DIR / 'vocab.json'

MAX_LEN    = 40
EMBED_DIM  = 256
BATCH_SIZE = 64
EPOCHS     = 20


### 1. Feature Extraction

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from src.cnn.utils import extract_features

cnn_encoder = InceptionV3(include_top=False, pooling='avg', weights='imagenet')
cnn_encoder.trainable=False

FEATURE_DIM = cnn_encoder.output_shape[-1]
print(f'Feature dim: {FEATURE_DIM}')

In [ ]:
all_image_paths = sorted(IMAGES_DIR.glob('*.jpg'))
print(f'Total images: {len(all_image_paths)}')

feat_file = MODEL_DIR / 'inception_features.npy'
feat_names_file = feat_file = MODEL_DIR / 'inception_features.json'

if feat_file.exist() and feat_names_file.exist():
    feats_arr = np.load(feat_file)

    with open(feat_names_file) as f:
        feat_names = json.load(f)
        
    features = {}
    for name, feat in zip(feat_names, feats_arr):
        features[name] = feat

else:
    print('Extract feature process')

    paths_str = []
    for p in all_image_paths:
        paths_str.append(str(p))

    feats_arr = extract_features(paths_str, cnn_encoder, str(feat_file), target_size=(299, 299), preprocess_fn=preprocess_input, batch_size=64)
    
    feat_names = []
    for p in all_image_paths:
        feat_names.append(p.name)
    
    with open(feat_names_file, 'w') as f:
        json.dump(feat_names, f)

    features = {}
    for name, feat in zip(feat_names, feats_arr):
        features[name] = feat
    
    print(f'There is {len(features)} feats')

### 2. Caption Preprocessing

In [ ]:
from src.rnn_lstm.preprocess import (load_captions, 
                                     auto_split, 
                                     build_vocab, 
                                     save_vocab, 
                                     load_vocab, 
                                     make_sequences
                                     )

captions = load_captions(str(CAPS_CSV))
print(f'Total images: {len(captions)}')

train_imgs, val_imgs, test_imgs = auto_split(captions)
print(f'Split: {len(train_imgs)} train / {len(val_imgs)} val / {len(test_imgs)} test')


if VOCAB_PATH.exists():
    word2idx, idx2word = load_vocab(str(VOCAB_PATH))
else:
    word2idx = build_vocab(captions, train_imgs, min_freq=1)
    save_vocab(word2idx, str(VOCAB_PATH))

    idx2word = {}
    for k, v in word2idx.items():
        idx2word[v] = k

VOCAB_SIZE = len(word2idx)
print(f'Vocab size: {VOCAB_SIZE}')


In [ ]:
def make_tf_dataset(img_list, features_dict, captions_dict, word2idx, max_len, batch_size):
    imgs, cap_in, cap_tgt = make_sequences(captions_dict, img_list, word2idx, max_len)
    img_feats = np.stack([features_dict[img] for img in imgs], axis=0)

    weights = (cap_tgt != 0).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices(({'cap_input': cap_in, 'img_input': img_feats}, cap_tgt, weights)
                                            )
    return ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE), imgs, cap_tgt

train_ds, _, _  = make_tf_dataset(train_imgs, features, captions, word2idx, MAX_LEN, BATCH_SIZE)
val_ds, val_imgs_flat, val_cap_tgt = make_tf_dataset(val_imgs, features, captions, word2idx, MAX_LEN, BATCH_SIZE)
print('Dataset terbuat')

### 3. Decoder

In [ ]:
from tensorflow.keras.layers import(Input, Embedding, Dense, Reshape, Concatenate, LSTM, SimpleRNN, Lambda)
from tensorflow.keras.models import Model

def build_decoder(cell_type, num_layers, 
                  hidden_size, vocab_size=VOCAB_SIZE, 
                  embed_dim=EMBED_DIM, feature_dim=FEATURE_DIM, max_len=MAX_LEN):
    
    cap_in = Input(shape=(max_len,), name='cap_input')
    emb = Embedding(vocab_size, embed_dim, name='embedding')(cap_in)

    img_in = Input(shape=(feature_dim,), name='img_input')
    img_proj = Dense(embed_dim, name='dense_proj')(img_in)
    img_proj = Reshape((1, embed_dim))(img_proj)

    x = Concatenate(axis=1)([img_proj, emb])

    if cell_type == 'lstm':
        RNNLayer = LSTM
    else:
        RNNLayer = SimpleRNN

    for i in range(num_layers):
        x = RNNLayer(hidden_size, return_sequences=True,
                     name=f'{cell_type}_{i+1}')(x)

    x = Lambda(lambda t: t[:, 1:, :], name='slice_output')(x)
    out = Dense(vocab_size, activation='softmax', name='dense_out')(x)

    return Model([cap_in, img_in], out, name=f'dec_{cell_type}_L{num_layers}_H{hidden_size}')

In [ ]:
CONFIGS = list(itertools.product(['rnn', 'lstm'], [1, 2, 3], [128, 512]))
print(f'Total configs: {len(CONFIGS)}')
for ct, nl, hs in CONFIGS:
    print(f'  {ct:4s}  layers={nl}  hidden={hs}')

In [ ]:
train_histories = {}
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

for i, (cell_type, n_layers, h_size) in enumerate(CONFIGS):
    model_name = f'{cell_type}_L{n_layers}_H{h_size}'
    save_path = MODEL_DIR / f'{model_name}.h5'

    print(f'[{i+1:02d}/12] {model_name}', end=' ... ')

    if save_path.exists():
        continue
    else:
        model = build_decoder(cell_type, n_layers, h_size)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

        hist = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=[early_stop], verbose=0)
        model.save(save_path)

        train_histories[model_name] = hist.history
        print(f'trained {len(hist.history["loss"])} epochs')

with open(MODEL_DIR / 'train_histories.json', 'w') as f:
    json.dump(train_histories, f)
print('\nSemua model tersimpan.')


NameError: name 'tf' is not defined

### 4. Test Case

####   A. Variasi Arsitektur

In [ ]:
from src.rnn_lstm.model import CaptioningFromScratch

def compute_bleu4(refs_corpus, hyps_corpus):
    smooth_func = SmoothingFunction().method1

    return corpus_bleu(refs_corpus, hyps_corpus, weights=(0.25,)*4, smoothing_function=smooth_func)

def compute_meteor_avg(refs_list, hyps_list):
    scores = []
    for refs, hyp in zip(refs_list, hyps_list):
        s = meteor_score([r.split() for r in refs], hyp.split())
        scores.append(s)

    return float(np.mean(scores))

def eval_model_scratch(model_path, img_list, features_dict, captions_dict, word2idx, idx2word, max_len=MAX_LEN):
    decoder = tf.keras.models.load_model(model_path)
    scratch = CaptioningFromScratch.from_keras(cnn_encoder, decoder, img_size=(299,299), preprocess_fn=preprocess_input)

    img_unique = list(dict.fromkeys(img_list))
    hyps = []
    refs = []

    for img in img_unique:
        feat = features_dict.get(img)
        
        if feat is None:
            continue

        caption = scratch.generate_from_feature(feat, word2idx, idx2word, max_len)
        ref_raw = captions_dict.get(img, [])
        
        hyps.append(caption.split())
        refs.append([r.lower().split() for r in ref_raw]) 
     
    bleu4 = compute_bleu4(refs, hyps)
    meteor = compute_meteor_avg(
        [[' '.join(t) for t in rs] for rs in refs],
        [' '.join(h) for h in hyps]
    )

    return bleu4, meteor, scratch
    
    print('Helper functions siap.')

In [ ]:
exp1_results = []
best_models  = {}

for ct, nl, hs in CONFIGS:
    model_name = f'{ct}_L{nl}_H{hs}'
    save_path  = MODEL_DIR / f'{model_name}.h5'
    print(f'Evaluating {model_name} ...', end=' ')

    bleu4, meteor, _ = eval_model_scratch(
        save_path, test_imgs, features, captions, word2idx, idx2word  # test set
    )
    exp1_results.append({
        'model': model_name, 'cell': ct,
        'n_layers': nl, 'hidden': hs,
        'bleu4': round(bleu4, 4), 'meteor': round(meteor, 4)
    })
    print(f'BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}')

df1 = pd.DataFrame(exp1_results).sort_values('bleu4', ascending=False).reset_index(drop=True)
display(df1)

best_rnn_row  = df1[df1.cell=='rnn'].iloc[0]
best_lstm_row = df1[df1.cell=='lstm'].iloc[0]
print(f'Best RNN : {best_rnn_row["model"]} (BLEU-4={best_rnn_row["bleu4"]:.4f})')
print(f'Best LSTM: {best_lstm_row["model"]} (BLEU-4={best_lstm_row["bleu4"]:.4f})')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, title) in zip(axes, [
    ('cell', 'Tipe Cell (RNN vs LSTM)'),
    ('n_layers', 'Jumlah Layer'),
    ('hidden',   'Hidden Size'),
]):
    grp = df1.groupby(col)['bleu4'].mean().sort_values(ascending=False)
    ax.bar(grp.index.astype(str), grp.values, color='coral')
    ax.set_title(title); ax.set_ylabel('Mean BLEU-4')
    ax.set_ylim(max(0, grp.min()-0.01), grp.max()+0.01)
    for j, v in enumerate(grp.values):
        ax.text(j, v+0.001, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Pengaruh Hyperparameter terhadap BLEU-4')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'exp1_hparam.png', bbox_inches='tight')
plt.show()

In [ ]:
try:
    with open(MODEL_DIR / 'train_histories.json') as f:
        loaded_hist = json.load(f)
    train_histories.update(loaded_hist)
except FileNotFoundError:
    pass

if train_histories:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in train_histories.items():
        ct = name.split('_')[0]
        ls = '-' if ct == 'lstm' else '--'
        ax1.plot(h['loss'],     label=name, linestyle=ls, alpha=0.7)
        ax2.plot(h['val_loss'], label=name, linestyle=ls, alpha=0.7)
    ax1.set_title('Train Loss'); ax1.legend(fontsize=6)
    ax2.set_title('Val Loss');   ax2.legend(fontsize=6)
    plt.suptitle('Loss Curves — Semua 12 Variasi')
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'exp1_loss_curves.png', bbox_inches='tight')
    plt.show()

####   B. Keras v. Scratch

In [2]:
eval_imgs_exp2 = []

for img in test_imgs:
    if img in features:
        eval_imgs_exp2.append(img)

eval_imgs_exp2 = eval_imgs_exp2[:100]

# --
eval_feats_exp2 = []
temp_feat_list = []

for img in eval_imgs_exp2:
    feat = features[img]
    temp_feat_list.append(feat)

eval_feats_exp2 = np.stack(temp_feat_list)

# --
eval_refs_exp2 = []

for img in eval_imgs_exp2:
    caption = captions.get(img, [])

    result_caption = []

    for capt in caption:
        lowered = capt.lower()
        split = lowered.split()

        sentence = ' '.join(split)
        result_caption.append(sentence)

    eval_refs_exp2.append(result_caption)


rnn_path  = MODEL_DIR / f'{best_rnn_row["model"]}.h5'
lstm_path = MODEL_DIR / f'{best_lstm_row["model"]}.h5'
rnn_decoder_e2  = tf.keras.models.load_model(rnn_path)
lstm_decoder_e2 = tf.keras.models.load_model(lstm_path)

NameError: name 'test_imgs' is not defined

In [ ]:
rnn_scratch_e2 = CaptioningFromScratch.from_keras(cnn_encoder, rnn_decoder_e2, 
                                                  img_size=(299, 299), preprocess_fn=preprocess_input)

lstm_scratch_e2 = CaptioningFromScratch.from_keras(cnn_encoder, lstm_decoder_e2,
                                                   img_size=(299, 299), preprocess_fn=preprocess_input)


time_start = time.time()
rnn_scratch_captions = rnn_scratch_e2.generate_batch(eval_feats_exp2, word2idx, idx2word, MAX_LEN)
time_rnn_scratch = time.time() - time_start

time_start = time.time()
lstm_scratch_captions = lstm_scratch_e2.generate_batch(eval_feats_exp2, word2idx, idx2word, MAX_LEN)
time_lstm_scratch = time.time() - time_start

print(f'RNN Scratch : {time_rnn_scratch:.2f}s')
print(f'LSTM Scratch : {time_lstm_scratch:.2f}s')

In [ ]:
def keras_greedy_decode(decoder_model, feat, word2idx, idx2word, max_len):
    start_id = word2idx['<start>']
    end_id = word2idx['<end>']

    tokens = np.zeros((1, max_len), dtype=np.int32)
    tokens[0, 0] = start_id
    feat_b = feat[np.newaxis]
    result = []

    for timestep in range(max_len -1):
        out = decoder_model.predict([tokens, feat_b], verbose = 0)
        next_token =  int(np.argmax(out[0, timestep]))

        if next_token == end_id:
            break
        
        result.append(next_token)
        tokens[0, timestep+1] = next_token
    
    list_words = []
    for r in result:
        word = idx2word.get(r, '<unk>')
        list_words.append(word)

    hasil = ' '.join(list_words)
    return hasil

time_start = time.time()

rnn_keras_captions = []
for feat in eval_feats_exp2:
    caption = keras_greedy_decode(rnn_decoder_e2, feat, word2idx, idx2word, MAX_LEN)
    
    rnn_keras_captions.append(caption)
    
time_rnn_keras = time.time() - time_start

lstm_keras_captions = []
time_start = time.time()
for feat in eval_feats_exp2:
    caption = keras_greedy_decode(lstm_decoder_e2, feat, word2idx, idx2word, MAX_LEN)
    lstm_keras_captions.append(caption)
    
time_lstm_keras = time.time() - time_start

print(f'RNN Keras: {time_rnn_keras}')
print(f'LSTM Keras: {time_lstm_keras}')

In [ ]:
def captions_to_refs_hyps(captions_raw, hyps_str):
    refs = [[r.split() for r in rs] for rs in captions_raw]
    hyps = [h.split() for h in hyps_str]
    
    return refs, hyps
        
rows = []
for label, caps, t in [
    (f'RNN  Keras   ({best_rnn_row["model"]})',    rnn_keras_captions,    time_rnn_keras),
    (f'RNN  Scratch ({best_rnn_row["model"]})',    rnn_scratch_captions,  time_rnn_scratch),
    (f'LSTM Keras   ({best_lstm_row["model"]})',   lstm_keras_captions,   time_lstm_keras),
    (f'LSTM Scratch ({best_lstm_row["model"]})',   lstm_scratch_captions, time_lstm_scratch),
    ]:
    
    refs, hyps = captions_to_refs_hyps(eval_refs_exp2, caps)
    bleu4 = compute_bleu4(refs, hyps)
    meteor_avg = compute_meteor_avg(eval_refs_exp2, caps)
    rows.append({'Variasi': label, 'BLEU-4': round(bleu4), 'METEOR': round(meteor_avg,4), 'Waktu(s)': round(t,2)})

df_exp2 = pd.DataFrame(rows)
display(df_exp2)

best_exp2_row = df_exp2.loc[df_exp2['BLEU-4'].idxmax()]
best_exp2_is_lstm = 'LSTM' in best_exp2_row['Variasi']

if best_exp2_is_lstm:
    best_exp2_cell_type = best_lstm_row
else:
    best_lstm_row = best_rnn_row
    
print(f'\nModel terbaik untuk Exp4: {best_exp2_row["Variasi"]}')

identical_rnn = 0
identical_lstm = 0
for rnn_kc, rnn_sc in zip(rnn_keras_captions, rnn_scratch_captions):
    if rnn_kc == rnn_sc:
        identical_rnn += 1
        
for lstm_kc, lstm_sc in zip(lstm_keras_captions, lstm_scratch_captions):
    if lstm_kc == lstm_sc:
        identical_lstm += 1

# print(f'\nKaption identik RNN  (Keras vs Scratch): {identical_rnn}/{len(eval_feats_exp2)}'
#       f' ({100*identical_rnn/len(eval_feats_exp2):.1f}%)')
# print(f'Kaption identik LSTM (Keras vs Scratch): {identical_lstm}/{len(eval_feats_exp2)}'
#       f' ({100*identical_lstm/len(eval_feats_exp2):.1f}%)')  

#### C. RNN v. LSTM

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score as single_meteor

best_rnn_path = MODEL_DIR / f'{best_rnn_row["model"]}.h5'
best_lstm_path = MODEL_DIR / f'{best_lstm_row["model"]}.h5'

rnn_decoder = tf.keras.models.load_model(best_rnn_path)
lstm_decoder = tf.keras.models.load_model(best_lstm_path)

rnn_scratch = CaptioningFromScratch.from_keras(cnn_encoder, rnn_decoder, preprocess_fn=preprocess_input)
lstm_scratch = CaptioningFromScratch.from_keras(cnn_encoder, lstm_decoder, preprocess_fn=preprocess_input)

qual_imgs = []
for image in test_imgs[:50]:
    if image in features:
        qual_imgs.append(image)
        
per_imgs = []
time_rnn_total = 0.0
time_lstm_total = 0.0

for image in qual_imgs:
    feature = features[image]
    ref_raw = captions.get(image, [])
    smoother = SmoothingFunction().method1
    
    ref_tokens = []
    for ref in ref_raw:
        lowered = ref.lower()
        split = lowered.split()
        
        ref_tokens.append(split)
    
    time_start = time.time()
    rnn_cap = rnn_scratch.generate_from_feature(feature, word2idx, idx2word, MAX_LEN)
    time_rnn_total += time.time() - time_start
    
    time_start = time.time()
    lstm_cap = lstm_scratch.generate_from_feature(feature, word2idx, idx2word, MAX_LEN)
    time_lstm_total += time.time() - time_start
    
    
    bleu_rnn = sentence_bleu(ref_tokens, rnn_cap.split(), weights=(0.25,)*4, smoothing_function=smoother)
    bleu_lstm = sentence_bleu(ref_tokens, lstm_cap.split(), weights=(0.25,)*4, smoothing_function=smoother)
    
    if ref_raw:
        ref_split = []
    
        for ref in ref_raw:
            ref_split.append(ref.split())
            
        rnn_split = rnn_cap.split()
        lstm_split = lstm_cap.split()
        
        meteor_rnn = single_meteor(ref_split, rnn_split)
        meteor_lstm = single_meteor(ref_split, lstm_split)

    else:
        meteor_rnn = 0.0
        meteor_lstm = 0.0
    
    per_imgs.append({
        'img': img, 'rnn': rnn_cap, 'lstm': lstm_cap,
        'bleu_rnn': bleu_rnn, 'bleu_lstm': bleu_lstm,
        'meteor_rnn': meteor_rnn, 'meteor_lstm': meteor_lstm,
        'refs': [r.lower() for r in ref_raw],
    })
    
    df_qual = pd.DataFrame(per_imgs).sort_values('bleu_lstm', ascending=False).reset_index(drop=True)
    

    

In [ ]:
n = len(df_qual)
high = df_qual.head(4)
mid = df_qual.iloc[n//2 - 1 : n//2 + 2]
low = df_qual.tail(3)
showcase = pd.concat([high, mid, low]).drop_duplicates('img').reset_index(drop=True)

for _, row in showcase.iterrows():
    img_path = IMAGES_DIR / row['img']
    print(f"Image: {row['img']}")
    print(f"Ref: {row['refs'][0]}")
    print(f"RNN: {row['rnn']} (BLEU={row['bleu_rnn']:.4f})")
    print(f"LSTM: {row['lstm']} (BLEU={row['bleu_lstm']:.4f})")
    
    if img_path.exists():
        from PIL import Image
        img_arr = np.array(Image.open(img_path).resize((244, 244)))
        plt.figure(figsize=(4, 3))
        plt.imshow(img_arr)
        plt.axis('off')
        
        plt.title(f'LSTM BLEU={row["bleu_lstm"]:.3f}', fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

lim = max(df_qual[['bleu_rnn','bleu_lstm']].max()) * 1.1

axes[0].scatter(df_qual['bleu_rnn'], df_qual['bleu_lstm'], alpha=0.5)
axes[0].plot([0, lim], [0, lim], 'r--', label='RNN = LSTM')
axes[0].set_xlabel('BLEU-4 SimpleRNN')
axes[0].set_ylabel('BLEU-4 LSTM')
axes[0].set_title('Per-image BLEU-4: RNN vs LSTM'); axes[0].legend()

lim_m = max(df_qual[['meteor_rnn','meteor_lstm']].max()) * 1.1
axes[1].scatter(df_qual['meteor_rnn'], df_qual['meteor_lstm'], alpha=0.5, color='green')
axes[1].plot([0, lim_m], [0, lim_m], 'r--', label='RNN = LSTM')
axes[1].set_xlabel('METEOR SimpleRNN')
axes[1].set_ylabel('METEOR LSTM')
axes[1].set_title('Per-image METEOR: RNN vs LSTM'); axes[1].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'exp3_rnn_vs_lstm.png', bbox_inches='tight')
plt.show()

print(f'{"Metrik":25s} {"SimpleRNN":>12s} {"LSTM":>12s}')
print('-' * 50)
print(f'{"Mean BLEU-4":25s} {df_qual["bleu_rnn"].mean():>12.4f} {df_qual["bleu_lstm"].mean():>12.4f}')
print(f'{"Mean METEOR":25s} {df_qual["meteor_rnn"].mean():>12.4f} {df_qual["meteor_lstm"].mean():>12.4f}')
print(f'{"Avg time/image (ms)":25s} {time_rnn_total/len(qual_imgs)*1000:>12.1f} {time_lstm_total/len(qual_imgs)*1000:>12.1f}')


#### D. Variasi Caption

In [ ]:
MAX_LEN_VARIANTS = [20, 40, 60]
exp4_result = []

if best_exp2_is_lstm:
    exp4_cell = 'lstm'
else:
    exp4_cell = 'rnn'       
    
exp4_config = best_exp2_cell_type

test_imgs_exp4 = []
for test_img in test_imgs:
    if test_img in features:
        test_imgs_exp4.append(img)
        
test_imgs_exp4 = test_imgs_exp4[:100]

temp = []
for img in test_imgs_exp4:
    data_fitur = features[img]
    temp.append(data_fitur)
    
test_feats_exp4 = np.stack(temp)
    
test_refs_exp4 = []
for img in test_imgs_exp4:
    raw_caption = captions.get(img, [])
    capt = []
    
    for caption in raw_caption:
        word = caption.lower().split()
        capt.append(word)
    
    test_refs_exp4.append(capt)

for max_len_var in MAX_LEN_VARIANTS:
    save_path = MODEL_DIR / f'best_{exp4_cell}_maxlen{max_len_var}.h5'
    
    if save_path.exists():
        model = tf.keras.models.load_model(save_path)
    else:
        train_ds, _, _ = make_tf_dataset(train_imgs, features, captions, word2idx, max_len_var, BATCH_SIZE)
        val_ds, _, _ = make_tf_dataset(val_imgs, features, captions, word2idx, max_len_var, BATCH_SIZE)
        
        model = build_decoder( exp4_cell, exp4_config['n_layers'], exp4_config['hidden'], max_len=max_len_var)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=[early_stop], verbose=0)
        model.save(save_path)
    
    sc = CaptioningFromScratch.from_keras(cnn_encoder, model, preprocess_fn=preprocess_input)
    hyps = []
    for feat in test_feats_exp4:
        gen = sc.generate_from_feature(feat, word2idx, idx2word, max_len_var).split()
        hyps.append(gen)
        
    bleu4 = compute_bleu4(test_refs_exp4, hyps)
    avg_len = np.mean([len(h) for h in hyps])
    exp4_result.append({'max_len': max_len_var, 'bleu4': round(bleu4,4), 'avg_cap_len': round(avg_len,1)})

df4 = pd.DataFrame(exp4_result)
display(df4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df4['max_len'], df4['bleu4'], 'o-', color='steelblue', label='BLEU-4')
ax2 = ax.twinx()
ax2.plot(df4['max_len'], df4['avg_cap_len'], 's--', color='coral', label='Avg caption len')
ax.set_xlabel('max_len')
ax.set_ylabel('BLEU-4')
ax2.set_ylabel('Avg Caption Length')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Pengaruh max_len terhadap BLEU-4 dan Panjang Caption')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'exp4_maxlen.png', bbox_inches='tight')
plt.show()
print('Analisis: max_len terlalu kecil → caption terpotong → BLEU rendah.')
print('max_len terlalu besar → model sulit belajar, lebih sering padding → BLEU bisa drop juga.')

### 5. Bonus - Init-Inject vs Pre-Inject

#### 5A. Build & Train Init-Inject Decoder

In [ ]:
def build_decoder_init_inject(cell_type, num_layers, hidden_size,
                               vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                               feature_dim=FEATURE_DIM, max_len=MAX_LEN):
    cap_in = Input(shape=(max_len,), name='cap_input')
    emb = Embedding(vocab_size, embed_dim, name='embedding')(cap_in)

    img_in = Input(shape=(feature_dim,), name='img_input')
    img_proj = Dense(hidden_size, name='dense_proj')(img_in) # (B, hidden)
    img_proj_rep = RepeatVector(max_len, name='repeat')(img_proj) # (B, T, hidden)

    RNNLayer = LSTM if cell_type == 'lstm' else SimpleRNN

    x = emb
    for i in range(num_layers):
        x = RNNLayer(hidden_size, return_sequences=True,
                     name=f'{cell_type}_{i+1}')(x) # (B, T, hidden)

    x = Add(name='add_context')([x, img_proj_rep]) # (B, T, hidden)
    out = Dense(vocab_size, activation='softmax', name='dense_out')(x)

    return Model([cap_in, img_in], out,
                 name=f'ii_{cell_type}_L{num_layers}_H{hidden_size}')

temp = build_decoder_init_inject('lstm', 1, 128)
temp.summary()

In [ ]:
ii_histories = {}
MODEL_DIR_II = MODEL_DIR

for i, (cell_type, n_layers, h_size) in enumerate(CONFIGS):
    model_name = f'ii_{cell_type}_L{n_layers}_H{h_size}'
    save_path  = MODEL_DIR_II / f'{model_name}.h5'
    print(f'[{i+1:02d}/12] {model_name}', end=' ... ')

    if save_path.exists():
        print('loaded')
        continue

    model = build_decoder_init_inject(cell_type, n_layers, h_size)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    hist = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                     callbacks=[early_stop], verbose=0)
    model.save(save_path)
    ii_histories[model_name] = hist.history
    print(f'trained {len(hist.history[loss])} epochs')

with open(MODEL_DIR / 'ii_histories.json', 'w') as f:
    json.dump(ii_histories, f)
print('Semua init-inject model tersimpan.')

#### 5B. Evaluasi Init-Inject

In [ ]:
ii_results = []

for ct, nl, hs in CONFIGS:
    model_name = f'ii_{ct}_L{nl}_H{hs}'
    save_path = MODEL_DIR / f'{model_name}.h5'
    print(f'Evaluating {model_name} ...', end=' ')

    bleu4, meteor, _ = eval_model_scratch(
        save_path, test_imgs, features, captions, word2idx, idx2word
    )
    ii_results.append({
        'model': model_name, 'arch': 'init_inject', 'cell': ct,
        'n_layers': nl, 'hidden': hs,
        'bleu4': round(bleu4, 4), 'meteor': round(meteor, 4)
    })
    print(f'BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}')

df_ii = pd.DataFrame(ii_results).sort_values('bleu4', ascending=False).reset_index(drop=True)
display(df_ii)

#### 5C. Perbandingan Pre-Inject vs Init-Inject (BLEU-4)

In [ ]:
df_inject = pd.DataFrame(exp1_results).assign(arch='pre_inject')
df_init_inj = pd.DataFrame(ii_results)

df_compare = pd.concat([df_inject, df_init_inj], ignore_index=True)

# Mean BLEU-4
summary = df_compare.groupby('arch')['bleu4'].agg(['mean','max']).round(4)
print(summary)

# Bar chart perbandingan
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col in zip(axes, ['cell', 'hidden']):
    grp = df_compare.groupby([col, 'arch'])['bleu4'].mean().unstack()
    grp.plot(kind='bar', ax=ax, colormap='Set2', width=0.6)
    ax.set_title(f'BLEU-4 by {col}')
    ax.set_ylabel('Mean BLEU-4')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='arch', fontsize=8)

plt.suptitle('Pre-Inject vs Init-Inject — Pengaruh Hyperparameter')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'bonus_inject_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter
merged = pd.merge(
    df_inject[['cell','n_layers','hidden','bleu4']].rename(columns={'bleu4':'bleu_pre'}),
    df_init_inj[['cell','n_layers','hidden','bleu4']].rename(columns={'bleu4':'bleu_ii'}),
    on=['cell','n_layers','hidden']
)

fig, ax = plt.subplots(figsize=(6, 5))
colors = ['steelblue' if ct == 'lstm' else 'coral' for ct in merged['cell']]
ax.scatter(merged['bleu_pre'], merged['bleu_ii'], c=colors, s=80, alpha=0.8)

lim = max(merged[['bleu_pre','bleu_ii']].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', alpha=0.5, label='same BLEU')
ax.set_xlabel('BLEU-4 Pre-Inject')
ax.set_ylabel('BLEU-4 Init-Inject')
ax.set_title('Head-to-Head: Pre-Inject vs Init-Inject (biru=LSTM, merah=RNN)')
ax.legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'bonus_inject_scatter.png', bbox_inches='tight')
plt.show()

win_ii  = (merged['bleu_ii'] > merged['bleu_pre']).sum()
win_pre = (merged['bleu_pre'] >= merged['bleu_ii']).sum()
print(f'Init-Inject menang di {win_ii}/12 config')
print(f'Pre-Inject menang di {win_pre}/12 config')

### 6. Bonus - Beam Search Decoder

#### 6A. Greedy vs Beam Search (k=3, k=5)

In [ ]:
import time
from src.rnn_lstm.model import CaptioningFromScratch

# Pakai model terbaik 
best_model_name = best_lstm_row["model"]
best_path = MODEL_DIR / f"{best_model_name}.h5"
best_decoder = tf.keras.models.load_model(best_path)
scratch = CaptioningFromScratch.from_keras(cnn_encoder, best_decoder,
                                           img_size=(299,299),
                                           preprocess_fn=preprocess_input)

# Sample 100 gambar 
eval_imgs_beam = [img for img in test_imgs if img in features][:100]
eval_feats_beam = np.stack([features[img] for img in eval_imgs_beam])
eval_refs_beam = [
    [c.lower().split() for c in captions.get(img, [])]
    for img in eval_imgs_beam
]

print(f"Evaluasi {len(eval_imgs_beam)} gambar dengan 3 strategi decoding...")

In [ ]:
beam_results = {}

for label, k in [("greedy", None), ("beam_k3", 3), ("beam_k5", 5)]:
    t0 = time.time()

    if k is None:
        hyps = scratch.generate_batch(eval_feats_beam, word2idx, idx2word, MAX_LEN)
    else:
        hyps = scratch.generate_batch_beam(eval_feats_beam, word2idx, idx2word, MAX_LEN, k=k)

    elapsed = time.time() - t0
    hyps_tok = [h.split() for h in hyps]
    bleu4 = compute_bleu4(eval_refs_beam, hyps_tok)
    meteor = compute_meteor_avg(
        [[" ".join(r) for r in rs] for rs in eval_refs_beam], hyps
    )

    beam_results[label] = {"bleu4": round(bleu4, 4), "meteor": round(meteor, 4),
                           "time": round(elapsed, 2), "hyps": hyps}
    print(f"{label:10s}  BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}  ({elapsed:.1f}s)")

#### 6B. Tabel & Visualisasi Perbandingan

In [ ]:
df_beam = pd.DataFrame([
    {"decoding": k, **{m: v for m,v in vals.items() if m != "hyps"}}
    for k, vals in beam_results.items()
])
display(df_beam)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [("bleu4", "BLEU-4"), ("meteor", "METEOR"), ("time", "Waktu (s)")]

for ax, (col, title) in zip(axes, metrics):
    vals = [beam_results[d][col] for d in ["greedy","beam_k3","beam_k5"]]
    bars = ax.bar(["Greedy","Beam k=3","Beam k=5"], vals,
                  color=["steelblue","coral","seagreen"])
    ax.set_title(title)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f"{v}", ha="center", fontsize=9)

plt.suptitle(f"Greedy vs Beam Search — {best_model_name}")
plt.tight_layout()
plt.savefig(MODEL_DIR / "bonus_beam_comparison.png", bbox_inches="tight")
plt.show()

#### 6C. Contoh Caption: Greedy vs Beam Search

In [ ]:
showcase_idx = list(range(min(5, len(eval_imgs_beam))))

for idx in showcase_idx:
    img_name = eval_imgs_beam[idx]
    img_path = IMAGES_DIR / img_name
    ref_text = captions.get(img_name, [""])[0]

    print(f"Image : {img_name}")
    print(f"Ref   : {ref_text.lower()}")
    print(f"Greedy: {beam_results['greedy']['hyps'][idx]}")
    print(f"k=3   : {beam_results['beam_k3']['hyps'][idx]}")
    print(f"k=5   : {beam_results['beam_k5']['hyps'][idx]}")

    if img_path.exists():
        from PIL import Image as PILImage
        img_arr = np.array(PILImage.open(img_path).resize((224, 224)))
        plt.figure(figsize=(3, 3))
        plt.imshow(img_arr); plt.axis("off")
        plt.tight_layout(); plt.show()
    print("-" * 60)
